# 02 – Phase A design evidence

Each section has a finding from the data, the code that shows it, and the Phase A design choice it supports.
The status of each choice is in the summary table at the end and in `docs/decisions.md`.

The normalization here is deliberately rough and only used to measure. The pipeline implementation lives in `src/normalize.py`.

In [1]:
import re
import unicodedata

import pandas as pd
from rapidfuzz import fuzz

pd.set_option("display.max_colwidth", 60)

df = pd.read_excel("../data/raw/presales_data_sample.xlsx")
inp = df.drop_duplicates("input_row_key")
print(f"{len(df)} candidate rows, {len(inp)} inputs")

2951 candidate rows, 592 inputs


## 1. Legal suffixes in client names

Client names end in many legal-form variants, often country-specific.

In [2]:
LEGAL_TOKENS = {"LTD", "LIMITED", "BHD", "BERHAD", "INC", "LLC", "LLP", "CO", "GMBH", "AG", "SA", "OY", "OYJ",
                "AB", "AS", "A/S", "APS", "ASA", "BV", "NV", "PLC", "PTE", "CORP", "COMPANY",
                "SDN", "PVT", "PUBL"}

def ending(name):
    toks = re.sub(r"[.,()]", " ", str(name)).upper().split()
    return " ".join(toks[-2:]) if len(toks) > 1 and toks[-1] in LEGAL_TOKENS else toks[-1]

inp.input_company_name.map(ending).value_counts().head(20)

input_company_name
PTE LTD                  43
SDN BHD                  39
CO LTD                   29
PRIVATE LIMITED          15
DANMARK A/S              15
NORWAY AS                13
NORGE AS                  9
SWEDEN AB                 8
DENMARK APS               8
DENMARK A/S               7
SVERIGE AB                6
IRELAND LIMITED           6
PVT LIMITED               6
V                         6
UK LIMITED                5
INTERNATIONAL LIMITED     5
SERVICES AS               4
A                         4
BANGLADESH LTD            4
COMPANY LIMITED           4
Name: count, dtype: int64

**Proposed:** split each name into core name and legal form before comparing names, and keep the legal form as its own layer.
Variants map to a canonical form and country in a reviewable lookup table, `data/reference/legal_forms.csv`.

## 2. Country words inside names

Many endings carry a country word before the legal form ("DANMARK A/S", "NORWAY AS"). These usually mark the local subsidiary of a multinational.

In [3]:
COUNTRY_WORDS = ["DANMARK", "DENMARK", "NORGE", "NORWAY", "SVERIGE", "SWEDEN", "SUOMI", "FINLAND",
                 "UK", "IRELAND", "SINGAPORE", "MALAYSIA", "PAKISTAN", "BANGLADESH", "THAILAND", "INDIA", "CHINA"]
pattern = r"\b(" + "|".join(COUNTRY_WORDS) + r")\b"
words = inp.input_company_name.str.upper().str.extract(pattern)[0]
print(f"{words.notna().sum()} of {len(inp)} input names contain a country word")
words.value_counts()

153 of 592 input names contain a country word


0
DANMARK       21
DENMARK       16
SINGAPORE     15
THAILAND      14
NORWAY        14
MALAYSIA      14
NORGE         11
PAKISTAN       9
SWEDEN         8
IRELAND        8
SVERIGE        7
UK             7
BANGLADESH     6
CHINA          2
FINLAND        1
Name: count, dtype: int64

**Proposed:** remove the country word from the core name for comparison, but keep it as a separate field. It can explain why the right candidate is a local entity of a global brand.

## 3. Norwegian postcodes lost their leading zero

Norwegian postcodes have 4 digits. Values under 1000 were stored as numbers, so "0668" became "668".

In [4]:
def clean_pc(s):
    return s.astype(str).str.replace(r"\.0$", "", regex=True).str.replace(" ", "")

no_in = clean_pc(inp.loc[inp.input_main_country_code == "NO", "input_main_postcode"].dropna())
no_cand = clean_pc(df.loc[df.main_country_code == "NO", "main_postcode"].dropna())
print(f"input:     {(no_in.str.len() < 4).sum()} of {len(no_in)} NO postcodes shorter than 4 digits")
print(f"candidate: {(no_cand.str.len() < 4).sum()} of {len(no_cand)} NO postcodes shorter than 4 digits")

input:     47 of 90 NO postcodes shorter than 4 digits
candidate: 129 of 296 NO postcodes shorter than 4 digits


**Proposed:** strip spaces and zero-pad postcodes to the country's length (lookup table) on both sides before comparing.

## 4. Administrative words in city names

Cities appear with administrative suffixes, on both the input and the candidate side ("Aalborg" vs "Aalborg Municipality"). Input counts are per input; candidate counts are per candidate row.

In [5]:
ADMIN_WORDS = r"Municipality|Kommune|Division|District"
for col in ["input_main_city", "main_city"]:
    s = (inp if col.startswith("input") else df)[col].dropna().astype(str)
    print(f"{col:16} {s.str.contains(ADMIN_WORDS, case=False).sum():4} of {len(s)} contain an administrative word")

input_main_city    58 of 515 contain an administrative word
main_city         153 of 2621 contain an administrative word


**Proposed:** remove administrative words, lowercase and fold accents (Ø→o) before comparing cities.

## 5. Missing fields

A missing value is not a mismatch. This decides whether a layer can be "unknown".

In [6]:
fill = pd.DataFrame({
    "input (per input)": inp[["input_main_city", "input_main_postcode", "input_main_street",
                              "input_main_street_number"]].notna().mean().values,
    "candidate (per row)": df[["main_city", "main_postcode", "main_street", "main_street_number"]].notna().mean().values,
}, index=["city", "postcode", "street", "street number"])
fill.loc["legal name"] = [None, df.company_legal_names.notna().mean()]
fill.loc["country"] = [inp.input_main_country_code.notna().mean(), df.main_country_code.notna().mean()]
fill.style.format("{:.0%}", na_rep="n/a")

,input (per input),candidate (per row)
city,87%,89%
postcode,82%,83%
street,66%,74%
street number,58%,65%
legal name,n/a,53%
country,100%,95%


**Proposed:**
- every layer has an "unknown" outcome, distinct from "different";
- legal form is "unknown" whenever the candidate has no legal name (about half the rows);
- because about a third of inputs have no street, rule R2 allows an auto-match on name and city when the input has no street, but only if the fail-safe check passes (approved answer 2).

## 6. Subset names: loose comparison is too loose, strict comparison too strict

`token_set_ratio` returns 100 whenever one name's words are a subset of the other's. `token_sort_ratio` compares the full word sets and penalizes every extra word.
The pairs below score 100 on the first and under 70 on the second. They are a mix:
- correct: KLM-ROYAL DUTCH AIRLINES Filial av utenlandsk foretak vs KLM, TRANSMODE SYSTEMS vs Transmode
- wrong: AIA Berhad vs AIA Public Takaful Sungai Buloh Medical Card, SiPM vs SIPM - Société IHEB des Portes Motorisées

In [7]:
LEGAL_RE = re.compile(r"\b(" + "|".join(re.escape(t) for t in sorted(LEGAL_TOKENS, key=len, reverse=True)) + r")\b")

def core(name):
    s = unicodedata.normalize("NFKD", str(name)).encode("ascii", "ignore").decode().upper()
    s = re.sub(r"[^\w/ ]", " ", s)
    return " ".join(LEGAL_RE.sub(" ", s).split())

pairs = df[["input_row_key", "input_company_name", "company_name"]].copy()
pairs["set"] = [fuzz.token_set_ratio(core(a), core(b)) for a, b in zip(pairs.input_company_name, pairs.company_name)]
pairs["sort"] = [fuzz.token_sort_ratio(core(a), core(b)) for a, b in zip(pairs.input_company_name, pairs.company_name)]
false_100 = pairs[(pairs.set == 100) & (pairs.sort < 70)]
print(f"{len(false_100)} candidate rows score 100 on token_set but under 70 on token_sort")
print(f"{false_100.input_row_key.nunique()} inputs affected")
false_100.sample(15, random_state=3)

506 candidate rows score 100 on token_set but under 70 on token_sort
286 inputs affected


,input_row_key,input_company_name,company_name,set,sort
1193,312,MAKIO ApS,Makio Castle Ruins,100.0,43.478261
788,24,AIA Berhad,Info Plan Takaful AIA,100.0,25.000000
2678,582,ZALARIS HR SERVICES NORWAY AS,Zalaris,100.0,42.424242
1767,417,Radio Frequency Systems Gesellschaft mit beschrankter Ha...,Radio Frequency Systems,100.0,54.761905
1423,355,Nokia Innovations Oy,Nokia Oyj,100.0,45.454545
181,130,DELOITTE ADVOKATFIRMA AS,Deloitte,100.0,55.172414
2604,569,WIDEROE AS,Wideroe Technical,100.0,58.333333
2509,551,TXO SYSTEMS LTD,Txo,100.0,42.857143
84,112,CRAYON AS,Crayon Software Asset,100.0,44.444444
1130,301,KROLL ASSOCIATES U.K. LIMITED,Kroll,100.0,40.000000


**Proposed:** compute both scores on core names. A subset match (`token_set` = 100 with a lower `token_sort`) gets its own name outcome, "subset", which never auto-matches on its own.
The name outcome cut-offs (exact / strong / weak) are chosen after looking at the score distribution.

## 6b. Branch markers in names

Some client names describe a branch of a foreign company ("Filial av utenlandsk foretak", "filial i Norge", "branch"). These words inflate the strict name score against the parent brand.

In [8]:
BRANCH = r"\bFILIAL\b|\bBRANCH\b|\bNUF\b|UTENLANDSK FORETAK"
has_branch = inp.input_company_name.str.upper().str.contains(BRANCH)
print(f"{has_branch.sum()} of {len(inp)} input names contain a branch marker")
inp.loc[has_branch, ["input_row_key", "input_company_name", "input_main_country_code"]]

11 of 592 input names contain a branch marker


,input_row_key,input_company_name,input_main_country_code
670,219,HELP Forsakring filial,SE
685,221,"HELP FORSIKRING,FILIAL AF HELP FORSIKRING AS, NORGE",DK
840,25,AIG EUROPE S.A. NUF,NO
875,256,"IKANO BANK, FILIAL AF IKANO BANK AB (PUBL), SVERIGE",DK
1080,293,KLM-ROYAL DUTCH AIRLINES Filial av utenlandsk foretak.,NO
1175,31,"AMAZON WEB SERVICES EMEA SARL, NORWEGIAN BRANCH",NO
1285,33,"American Express Europe Denmark, filial af American Expr...",DK
1442,359,"NORDEA BANK ABP, FILIAL I NORGE",NO
2201,497,"TATA CONSULTANCY SERVICES LIMITED, FILIAL AF TATA CONSUL...",DK
2276,51,"Ascio Technologies, Corp. Danmark - Filial af Ascio Tech...",DK


**Proposed:** remove branch markers from the core name and keep them as a separate field, like country words.

## 7. Several candidates often share the input's street

When two or more candidates sit on the input's street, an address match cannot pick a winner on its own.

In [9]:
def street_key(s):
    if pd.isna(s):
        return None
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    return " ".join(re.sub(r"[^\w ]", " ", s).split())

st = df.assign(i=df.input_main_street.map(street_key), c=df.main_street.map(street_key))
st["same_street"] = st.i.notna() & (st.i == st.c)
shared = st.groupby("input_row_key").same_street.sum()
print(f"{(shared >= 2).sum()} of {len(inp)} inputs have 2+ candidates on the input's exact street")
print(f"{(shared == 1).sum()} inputs have exactly one")

111 of 592 inputs have 2+ candidates on the input's exact street
136 inputs have exactly one


**Proposed:** the fail-safe check. After picking the best candidate, any other candidate with the same name outcome and the same or better location outcome sends the input to review (reason "rival").

## 8. "No candidate in country" is partly missing data

Candidates with no country cannot be in the input's country, so they inflate this group.

In [10]:
df["same_country"] = df.main_country_code == df.input_main_country_code
g = df.groupby("input_row_key").agg(any_same=("same_country", "any"),
                                    any_null=("main_country_code", lambda s: s.isna().any()))
none_same = g[~g.any_same]
print(f"{len(none_same)} inputs have no candidate in the input's country")
print(f"{none_same.any_null.sum()} of them have at least one candidate with no country")

26 inputs have no candidate in the input's country
13 of them have at least one candidate with no country


**Proposed:** country outcome is "unknown" (not "different") when the candidate has no country, and rule R3 (no candidate) treats unknown country as possibly the same.

## Summary

| # | Finding | Proposed design choice | Status |
|---|---|---|---|
| 1 | Many legal-form variants | Split core name / legal form; legal form is a layer; lookup table | approved (D6) |
| 2 | Country words mark local subsidiaries | Remove from core name, keep as field | approved (D6) |
| 3 | NO postcodes lost leading zeros | Zero-pad by country | approved (D6) |
| 4 | Administrative words in cities | Strip, lowercase, fold accents | approved (D6) |
| 5 | Many missing fields | "unknown" outcome; R2 for inputs without street | approved (D5, D6) |
| 6 | Subset names are a mix of right and wrong | Strict score first, looser fallback weighted lower; "subset" outcome | approved in principle (D7); subset strictness after A1 (D8) |
| 6b | Branch markers in names | Remove from core name, keep as field | approved (D6) |
| 7 | Shared streets are common | Fail-safe rival check | approved (D6) |
| 8 | Null countries inflate "no candidate" | Country unknown ≠ different | approved (D6) |